# Rebalancer experiment: baseline vs treatment

Compares two simulator runs side-by-side, identical in every respect except the rebalancer:

- **Baseline:** historical pipeline with `LatentDemandInflatorPhase` to artificially raise demand and `DeparturePhysicsPhase(mode="strict")` so lost demand can appear in the log. **No rebalancer.**
- **Treatment:** identical pipeline + `DispatchPhase(RebalancerTask, schedule=Schedule.every_n(N))` appended at the end. Same demand multiplier, same seed.

Both runs share the first six phases:

1. `HistoricalLatentDemandPhase` — publishes O_i, D_j marginals
2. `LatentDemandInflatorPhase(multiplier=...)` — multiplies latent demand to force shortages
3. `HistoricalODStructurePhase` — publishes P(j | i)
4. `DeparturePhysicsPhase(mode="strict")` — clipped at inventory; gap → lost demand
5. `HistoricalTripSamplingPhase` — replays observed trips into in_transit
6. `ArrivalsPhase` — delivers in_transit, emits flow events

Treatment adds:

7. `DispatchPhase(RebalancerTask(...), schedule=Schedule.every_n(N))` — moves bikes between stations via trucks

Subsequent cells produce five comparison tables. Tweak `demand_multiplier`, `rebalance_every_n_periods`, and the `min_threshold`/`max_threshold` knobs of `RebalancerTask` to find a regime where treatment beats baseline.

## Block map

1. Imports
2. Build resolved mock model with trucks (parameters + `demand_multiplier` + `rebalance_every_n_periods`)
3. Baseline run (no rebalancer)
4. Treatment run (rebalancer every N periods)
5. Trips per date — comparison table
6. Trips per source station — comparison table
7. Lost demand — comparison table (the primary success metric)
8. Total distance / revenue proxy — comparison table
9. Combined summary table

In [1]:
# %% 1. Imports
import sys
from pathlib import Path

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

import dataclasses

import pandas as pd

from gbp.build.pipeline import build_model
from gbp.consumers.simulator import (
    ArrivalsPhase,
    DeparturePhysicsPhase,
    Environment,
    EnvironmentConfig,
    HistoricalLatentDemandPhase,
    HistoricalODStructurePhase,
    HistoricalTripSamplingPhase,
)
from gbp.consumers.simulator.built_in_phases import LatentDemandInflatorPhase
from gbp.consumers.simulator.dispatch_phase import DispatchPhase
from gbp.consumers.simulator.phases import Schedule
from gbp.consumers.simulator.tasks.rebalancer import RebalancerTask
from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig

imports_loaded = True

In [2]:
# %% 2. Build resolved mock model with trucks
mock_config = {
    "n_stations": 10,
    "n_depots": 2,
    "n_timestamps": 72,
    "time_freq": "h",
    "start_date": "2025-01-01",
    "ebike_fraction": 0.3,
    "depot_capacity": 200,
    "seed": 42,
}
mock = DataLoaderMock(mock_config, n_trucks=3, truck_capacity_bikes=20)
loader = DataLoaderGraph(mock, GraphLoaderConfig())
raw = loader.load()
resolved_full = build_model(raw)

# Drop supply: observed_flow already encodes trip events on its target side,
# pre-populating in_transit from supply would double-count arrivals.  Same
# rationale as notebook 09 cell 3.
resolved = dataclasses.replace(resolved_full, supply=None)
demand_multiplier = 3.0
rebalance_every_n_periods = 6
rebalancer_commodity = "electric_bike"

2026-05-01 19:46:17 [info     ] load_start                     loader=graph_core
2026-05-01 19:46:17 [debug    ] source_validated               loader=graph_core
2026-05-01 19:46:17 [debug    ] observed_flow_built            loader=graph_core rows=305
2026-05-01 19:46:17 [debug    ] observed_inventory_built       loader=graph_core rows=60
2026-05-01 19:46:17 [info     ] load_done                      facilities=12 loader=graph_core


In [3]:
# %% 3. Baseline run: inflated demand, no rebalancer
baseline_phases = [
    HistoricalLatentDemandPhase(),
    LatentDemandInflatorPhase(multiplier=demand_multiplier),
    HistoricalODStructurePhase(),
    DeparturePhysicsPhase(mode="strict"),
    HistoricalTripSamplingPhase(),
    ArrivalsPhase(),
]
baseline_log = Environment(
    resolved,
    EnvironmentConfig(phases=baseline_phases, seed=42, scenario_id="baseline"),
).run()
baseline_tables = baseline_log.to_dataframes()

In [4]:
# %% 4. Treatment run: same pipeline + RebalancerTask
rebalancer_task = RebalancerTask(
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
    commodity_category=rebalancer_commodity,
)
treatment_phases = [
    HistoricalLatentDemandPhase(),
    LatentDemandInflatorPhase(multiplier=demand_multiplier),
    HistoricalODStructurePhase(),
    DeparturePhysicsPhase(mode="strict"),
    HistoricalTripSamplingPhase(),
    ArrivalsPhase(),
    DispatchPhase(
        rebalancer_task,
        schedule=Schedule.every_n(rebalance_every_n_periods),
    ),
]
treatment_log = Environment(
    resolved,
    EnvironmentConfig(phases=treatment_phases, seed=42, scenario_id="treatment"),
).run()
treatment_tables = treatment_log.to_dataframes()

In [5]:
# %% 5. Trips per date - baseline vs treatment
_period_to_date = resolved.periods[["period_id", "start_date"]].copy()
_period_to_date["date"] = pd.to_datetime(_period_to_date["start_date"]).dt.date


def _trips_per_date(flow_log: pd.DataFrame) -> pd.DataFrame:
    if flow_log.empty:
        return pd.DataFrame(columns=["date", "trips"])
    real_flow = flow_log[flow_log["source_id"] != "EXT"]
    merged = real_flow.merge(_period_to_date, on="period_id", how="left")
    return (
        merged.groupby("date", as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "trips"})
    )


baseline_trips_per_date = _trips_per_date(baseline_tables["simulation_flow_log"])
treatment_trips_per_date = _trips_per_date(treatment_tables["simulation_flow_log"])
trips_per_date_comparison = (
    baseline_trips_per_date.rename(columns={"trips": "baseline"})
    .merge(
        treatment_trips_per_date.rename(columns={"trips": "treatment"}),
        on="date",
        how="outer",
    )
    .fillna(0.0)
    .sort_values("date")
    .reset_index(drop=True)
)
trips_per_date_comparison["delta"] = (
    trips_per_date_comparison["treatment"] - trips_per_date_comparison["baseline"]
)
trips_per_date_comparison

,date,baseline,treatment,delta
0,2025-01-01,166.0,173.0,7.0
1,2025-01-02,176.0,183.0,7.0
2,2025-01-03,138.0,138.0,0.0


In [6]:
# %% 6. Trips per source station - baseline vs treatment
def _trips_per_station(flow_log: pd.DataFrame) -> pd.DataFrame:
    if flow_log.empty:
        return pd.DataFrame(columns=["source_id", "trips_outgoing"])
    real_flow = flow_log[flow_log["source_id"] != "EXT"]
    return (
        real_flow.groupby("source_id", as_index=False)["quantity"]
        .sum()
        .rename(columns={"quantity": "trips_outgoing"})
    )


baseline_trips_per_station = _trips_per_station(baseline_tables["simulation_flow_log"])
treatment_trips_per_station = _trips_per_station(treatment_tables["simulation_flow_log"])
trips_per_station_comparison = (
    baseline_trips_per_station.rename(columns={"trips_outgoing": "baseline"})
    .merge(
        treatment_trips_per_station.rename(columns={"trips_outgoing": "treatment"}),
        on="source_id",
        how="outer",
    )
    .fillna(0.0)
)
trips_per_station_comparison["delta"] = (
    trips_per_station_comparison["treatment"]
    - trips_per_station_comparison["baseline"]
)
trips_per_station_comparison = trips_per_station_comparison.sort_values(
    "delta", ascending=False,
).reset_index(drop=True)
trips_per_station_comparison

,source_id,baseline,treatment,delta
0,station_5,53.0,59.0,6.0
1,station_2,50.0,54.0,4.0
2,station_8,41.0,45.0,4.0
3,station_1,51.0,51.0,0.0
4,station_10,42.0,42.0,0.0
5,station_3,46.0,46.0,0.0
6,station_4,59.0,59.0,0.0
7,station_6,47.0,47.0,0.0
8,station_7,46.0,46.0,0.0
9,station_9,45.0,45.0,0.0


In [7]:
# %% 7. Lost demand - baseline vs treatment (primary success metric)
def _total_lost(tables: dict) -> float:
    log = tables.get("simulation_lost_demand_log")
    if log is None or log.empty:
        return 0.0
    return float(log["lost"].sum())


baseline_total_lost = _total_lost(baseline_tables)
treatment_total_lost = _total_lost(treatment_tables)
lost_demand_reduction_pct = (
    100.0 * (baseline_total_lost - treatment_total_lost) / baseline_total_lost
    if baseline_total_lost > 0 else 0.0
)
lost_demand_comparison = pd.DataFrame(
    {
        "scenario": ["baseline", "treatment"],
        "total_lost_demand": [baseline_total_lost, treatment_total_lost],
        "delta_vs_baseline": [
            0.0,
            treatment_total_lost - baseline_total_lost,
        ],
        "reduction_pct_vs_baseline": [0.0, lost_demand_reduction_pct],
    },
)
lost_demand_comparison

,scenario,total_lost_demand,delta_vs_baseline,reduction_pct_vs_baseline
0,baseline,937.0,0.0,0.000000
1,treatment,939.0,2.0,-0.213447


In [8]:
# %% 8. Total distance and revenue proxy - baseline vs treatment
def _total_distance_km(
    flow_log: pd.DataFrame, distance_matrix: pd.DataFrame,
) -> float:
    if flow_log.empty or distance_matrix is None or distance_matrix.empty:
        return 0.0
    real_flow = flow_log[flow_log["source_id"] != "EXT"].copy()
    if real_flow.empty:
        return 0.0
    merged = real_flow.merge(
        distance_matrix[["source_id", "target_id", "distance"]],
        on=["source_id", "target_id"],
        how="left",
    )
    merged["distance"] = merged["distance"].fillna(0.0)
    return float((merged["quantity"] * merged["distance"]).sum())


baseline_distance = _total_distance_km(
    baseline_tables["simulation_flow_log"], resolved.distance_matrix,
)
treatment_distance = _total_distance_km(
    treatment_tables["simulation_flow_log"], resolved.distance_matrix,
)
distance_comparison = pd.DataFrame(
    {
        "scenario": ["baseline", "treatment"],
        "total_distance_km": [baseline_distance, treatment_distance],
        # Revenue proxy v1: distance.  Replace with a real pricing model later.
        "revenue_proxy": [baseline_distance, treatment_distance],
    },
)
distance_comparison

,scenario,total_distance_km,revenue_proxy
0,baseline,3627.576299,3627.576299
1,treatment,3747.568725,3747.568725


In [9]:
# %% 9. Combined summary - all metrics, baseline vs treatment
baseline_total_trips = float(
    trips_per_date_comparison["baseline"].sum(),
)
treatment_total_trips = float(
    trips_per_date_comparison["treatment"].sum(),
)
summary_table = pd.DataFrame(
    {
        "metric": [
            "total_trips",
            "total_lost_demand",
            "total_distance_km",
            "revenue_proxy",
        ],
        "baseline": [
            baseline_total_trips,
            baseline_total_lost,
            baseline_distance,
            baseline_distance,
        ],
        "treatment": [
            treatment_total_trips,
            treatment_total_lost,
            treatment_distance,
            treatment_distance,
        ],
    },
)
summary_table["delta"] = summary_table["treatment"] - summary_table["baseline"]
summary_table["delta_pct"] = (
    100.0 * summary_table["delta"] / summary_table["baseline"].replace(0, pd.NA)
).fillna(0.0)
summary_table

,metric,baseline,treatment,delta,delta_pct
0,total_trips,480.000000,494.000000,14.000000,2.916667
1,total_lost_demand,937.000000,939.000000,2.000000,0.213447
2,total_distance_km,3627.576299,3747.568725,119.992426,3.307785
3,revenue_proxy,3627.576299,3747.568725,119.992426,3.307785
